<a href="https://colab.research.google.com/github/pradyotqc/QSL_VQE/blob/main/VQE_SL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install qiskit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 1.9 MB/s eta 0:00:00


In [2]:
!pip install qiskit_aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 104.0 MB/s eta 0:00:00


In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import Estimator # Corrected import for Estimator
from scipy.optimize import minimize
import numpy as np

# Define the Hamiltonian: H = Z
hamiltonian = SparsePauliOp("Z")

# Define a single-parameter variational circuit
theta = Parameter("θ")
qc = QuantumCircuit(1)
qc.ry(theta, 0)
var_form = qc

# Initialize the Estimator primitive (using Aer's Estimator)
estimator = Estimator()

# Storage for tracking parameter updates
history = []
iteration_count = 0 # Initialize iteration counter

# Expectation value function
def expectation_value(theta_val):
    # Bind parameters to the circuit using assign_parameters()
    # assign_parameters returns a new circuit with the parameters assigned
    bound_circuit = var_form.assign_parameters({theta: theta_val[0]})

    # Use the Estimator to get the expectation value
    job = estimator.run(bound_circuit, hamiltonian)
    result = job.result()

    return result.values[0].real

# Callback to track progress
def callback(theta_val):
    global iteration_count # Declare iteration_count as global to modify it
    current_theta = theta_val[0]
    # Pass the array to expectation_value as minimize does
    energy = expectation_value(theta_val)

    # Compute parameter change
    if history:
        delta = np.abs(current_theta - history[-1]['theta'])
    else:
        delta = 0.0

    # Store and print update
    history.append({'theta': current_theta, 'energy': energy, 'delta': delta})
    print(f"Iteration: {iteration_count:03d}, Theta: {current_theta:.5f}, Energy: {energy:.5f}, Δθ: {delta:.5f}")
    iteration_count += 1 # Increment iteration count

# Run COBYLA optimization
result = minimize(
    fun=expectation_value,
    x0=[0.1],
    method='COBYLA',
    callback=callback,
    options={'maxiter': 100, 'disp': False}
)

# Final result
print("\nOptimization complete.")
print(f"Minimum eigenvalue (approx.): {result.fun:.5f}")
print(f"Optimal theta: {result.x[0]:.5f}")

Iteration: 000, Theta: 2.10000, Energy: -0.46875, Δθ: 0.00000
Iteration: 001, Theta: 4.10000, Energy: -0.61523, Δθ: 2.00000
Iteration: 002, Theta: 4.10000, Energy: -0.55469, Δθ: 0.00000
Iteration: 003, Theta: 3.60000, Energy: -0.90820, Δθ: 0.50000
Iteration: 004, Theta: 3.10000, Energy: -1.00000, Δθ: 0.50000
Iteration: 005, Theta: 3.10000, Energy: -1.00000, Δθ: 0.00000
Iteration: 006, Theta: 3.10000, Energy: -0.99805, Δθ: 0.00000
Iteration: 007, Theta: 3.10000, Energy: -1.00000, Δθ: 0.00000
Iteration: 008, Theta: 3.10000, Energy: -1.00000, Δθ: 0.00000
Iteration: 009, Theta: 3.10000, Energy: -1.00000, Δθ: 0.00000
Iteration: 010, Theta: 3.10000, Energy: -1.00000, Δθ: 0.00000
Iteration: 011, Theta: 3.10000, Energy: -0.99609, Δθ: 0.00000

Optimization complete.
Minimum eigenvalue (approx.): -1.00000
Optimal theta: 3.10000


In [4]:
result

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -1.0
       x: [ 3.100e+00]
    nfev: 12
   maxcv: 0.0

In [5]:
import numpy as np
from scipy.optimize import minimize
from qiskit.circuit import QuantumCircuit, Parameter
from qiskit_aer.primitives import Estimator
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import TwoLocal
import time

# --- 1. Define the 2-Qubit Ising Hamiltonian ---
# H = -ZZ - h(IX + XI), we'll set the transverse field h = 0.1
hamiltonian = SparsePauliOp.from_list([("ZZ", -1.0), ("IX", -0.1), ("XI", -0.1)])

# --- 2. Calculate Theoretical Ground Truth Values (New Section) ---
# Convert the Hamiltonian operator to its matrix representation
H_matrix = hamiltonian.to_matrix()

# Use a classical eigensolver to find the exact eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eigh(H_matrix)

# Find the minimum eigenvalue and its corresponding eigenvector
min_eigenvalue_idx = np.argmin(eigenvalues)
theoretical_energy = eigenvalues[min_eigenvalue_idx]
target_eigenvector = eigenvectors[:, min_eigenvalue_idx]
target_state = Statevector(target_eigenvector)


# --- 3. Define a Two-Qubit Variational Circuit (Ansatz) ---
num_qubits = hamiltonian.num_qubits
var_form = TwoLocal(num_qubits, 'ry', 'cx', 'linear', reps=2)
num_params = var_form.num_parameters

# --- 4. Initialize the Estimator with Higher Precision ---
estimator = Estimator(run_options={'shots': 10000})

# --- Global variables for tracking optimization ---
history = {}
iteration_count = 0

# --- 5. Define Helper Functions ---
def expectation_value(params):
    bound_circuit = var_form.assign_parameters(params)
    job = estimator.run([bound_circuit], [hamiltonian])
    result = job.result()
    return result.values[0].real

def callback(current_params):
    global iteration_count
    energy = expectation_value(current_params)
    history[iteration_count] = {'params': current_params, 'energy': energy}
    print(f"  Iter: {iteration_count:03d} | Energy: {energy:.6f}")
    iteration_count += 1

# --- 6. Run the Optimization with Different Optimizers ---
optimizers = ['COBYLA', 'SLSQP', 'L-BFGS-B']

# Use the SAME random initial point for all optimizers for a fair comparison.
print(f"Generating a fixed random initial point for all optimizers...")
np.random.seed(42) # Use a seed for reproducibility
initial_params = np.random.rand(num_params)
print(f"Initial Parameters: {np.round(initial_params, 5)}\n")

for optimizer_method in optimizers:
    print(f"\n{'='*60}")
    print(f" Running optimization with: {optimizer_method}")
    print(f"{'='*60}")

    history = {}
    iteration_count = 0
    start_time = time.time()

    result = minimize(
        fun=expectation_value,
        x0=initial_params,
        method=optimizer_method,
        callback=callback,
        options={'maxiter': 200, 'disp': False}
    )
    end_time = time.time()

    # --- 7. Calculate Fidelity and Display Enhanced Results ---
    # Get the quantum state prepared by the VQE with the optimal parameters
    final_circuit = var_form.assign_parameters(result.x)
    vqe_state = Statevector.from_instruction(final_circuit)

    # Calculate the fidelity between the VQE state and the true target state
    fidelity = target_state.inner(vqe_state)

    print(f"\n Optimization complete for {optimizer_method}.")
    print(f"   - VQE Minimum Energy: {result.fun:.6f}")
    print(f"   - State Fidelity:     {fidelity:.6f}")
    print(f"   - Optimal Parameters:   {np.round(result.x, 5)}")
    print(f"   - Total Evaluations:    {result.nfev}")
    print(f"   - Execution Time:       {end_time - start_time:.2f} seconds")

# --- 8. Display Final Ground Truth Information ---
print(f"\n{'='*60}")
print(" Ground Truth Comparison")
print(f"{'='*60}")
print(f"Theoretical Ground State Energy: {theoretical_energy.real:.6f}")
print(f"Target Ground State Vector:\n{np.round(target_state.data, 5)}")

/tmp/ipython-input-3645291479.py:29: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  var_form = TwoLocal(num_qubits, 'ry', 'cx', 'linear', reps=2)


Generating a fixed random initial point for all optimizers...
Initial Parameters: [0.37454 0.95071 0.73199 0.59866 0.15602 0.15599]


 Running optimization with: COBYLA
  Iter: 000 | Energy: -0.651140
  Iter: 001 | Energy: -0.643340
  Iter: 002 | Energy: -0.701500
  Iter: 003 | Energy: -0.848300
  Iter: 004 | Energy: -0.901120
  Iter: 005 | Energy: -0.911540
  Iter: 006 | Energy: -0.924380
  Iter: 007 | Energy: -0.958240
  Iter: 008 | Energy: -0.999500
  Iter: 009 | Energy: -1.015020
  Iter: 010 | Energy: -1.015260
  Iter: 011 | Energy: -1.013080
  Iter: 012 | Energy: -1.014700
  Iter: 013 | Energy: -1.013880
  Iter: 014 | Energy: -1.018640
  Iter: 015 | Energy: -1.019480
  Iter: 016 | Energy: -1.016100
  Iter: 017 | Energy: -1.014840
  Iter: 018 | Energy: -1.017320
  Iter: 019 | Energy: -1.015200
  Iter: 020 | Energy: -1.016360
  Iter: 021 | Energy: -1.018680
  Iter: 022 | Energy: -1.012380
  Iter: 023 | Energy: -1.017300
  Iter: 024 | Energy: -1.014540
  Iter: 025 | Energy: -1.016100

/tmp/ipython-input-3645291479.py:71: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result = minimize(


  Iter: 000 | Energy: -0.079620
  Iter: 001 | Energy: -0.089100
  Iter: 002 | Energy: -0.094360

 Optimization complete for L-BFGS-B.
   - VQE Minimum Energy: -0.092040
   - State Fidelity:     -0.723673+0.000000j
   - Optimal Parameters:   [0.37454 0.95071 0.73199 0.59866 0.15602 0.15599]
   - Total Evaluations:    182
   - Execution Time:       22.82 seconds

 Ground Truth Comparison
Theoretical Ground State Energy: -1.019804
Target Ground State Vector:
[-0.70367+0.j -0.06968+0.j -0.06968+0.j -0.70367+0.j]


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# # --- Parameters for the Bound ---
# S_total_geom = np.pi / 2
# M_params = 1 # Number of parameters in our Ry(theta) ansatz
# delta_theta_max_assumption = 0.7 # Assumed max Euclidean parameter step norm per iteration

# # --- Calculate Theoretical N_min Bound ---
# # Using the formula: N_min >= (2 * S_total) / (sqrt(M) * delta_theta_max)
# N_min_theoretical_bound = (2 * S_total_geom) / (np.sqrt(M_params) * delta_theta_max_assumption)
# print(f"Theoretical N_min lower bound >= {np.ceil(N_min_theoretical_bound):.0f} (for delta_theta_max = {delta_theta_max_assumption})")

# # --- VQE Simulation Setup ---
# # Hamiltonian H = sigma_z
# # Ansatz U(theta) = Ry(theta) = exp(-i*theta*sigma_y/2)
# # Initial state |0>
# # Target state |1> (energy -1 for H=sigma_z)

# def energy_function(theta):
#     """ E(theta) = <0| Ry_dag(theta) * sigma_z * Ry(theta) |0> = cos(theta) """
#     return np.cos(theta)

# def gradient_energy_function(theta):
#     """ dE/dtheta = -sin(theta) """
#     return -np.sin(theta)

# def state_vector(theta):
#     """ |Psi(theta)> = Ry(theta)|0> = [cos(theta/2), sin(theta/2)]^T """
#     return np.array([np.cos(theta/2), np.sin(theta/2)])

# def g11_fsm(theta):
#     """
#     For Ry(theta) = exp(-i*theta*sigma_y/2) acting on |0>,
#     the effective generator K_1 = sigma_y/2.
#     g_11(theta) = Var(K_1) = <K_1^2> - <K_1>^2
#                 = <(sigma_y/2)^2> - (<sigma_y/2>)^2
#                 = (1/4) * <sigma_y^2> - (1/4) * (<sigma_y>)^2
#                 = (1/4) * <I> - (1/4) * (<sigma_y>)^2
#     For |Psi(theta)> = Ry(theta)|0>, <sigma_y> = 0.
#     So, g_11(theta) = 1/4.
#     """
#     return 1/4.0

# # --- Simulation 1: Gradient Descent with a fixed learning rate ---
# print("\n--- Simulation 1: Gradient Descent (fixed learning rate) ---")
# eta_gd = 0.25 # Learning rate
# theta_0_gd = 0.1 # Initial parameter, slightly off from gradient = 0
# target_energy_gd = -1.0
# energy_tolerance_gd = 1e-4
# max_iterations_gd = 200

# theta_hist_gd = [theta_0_gd]
# dLk_hist_gd = [0.0]
# L_total_hist_gd = [0.0]
# energy_hist_gd = [energy_function(theta_0_gd)]

# N_actual_S_total_gd = -1
# N_actual_energy_gd = -1
# converged_S_total_gd = False
# converged_energy_gd = False

# current_theta_gd = theta_0_gd
# print(f"{'Iter':>4} | {'Theta':>7} | {'GradE':>7} | {'dTheta':>7} | {'dL_k':>7} | {'L_total':>7} | {'Energy':>7}")
# print("-" * 60)
# for k_gd in range(1, max_iterations_gd + 1):
#     grad_E_gd = gradient_energy_function(current_theta_gd)
#     delta_theta_k_gd = -eta_gd * grad_E_gd # GD step

#     # Calculate dL_k for this step
#     # dL_k = sqrt(g11) * |delta_theta_k| for M=1
#     # We know g11 = 1/4 for Ry(theta) in this simple case
#     dL_k_val_gd = np.sqrt(g11_fsm(current_theta_gd)) * np.abs(delta_theta_k_gd)

#     current_theta_gd += delta_theta_k_gd
#     # No np.mod, let theta evolve freely as target is pi for cos(theta)=-1

#     theta_hist_gd.append(current_theta_gd)
#     dLk_hist_gd.append(dL_k_val_gd)
#     L_total_hist_gd.append(L_total_hist_gd[-1] + dL_k_val_gd)
#     current_energy_gd = energy_function(current_theta_gd)
#     energy_hist_gd.append(current_energy_gd)

#     print(f"{k_gd:>4} | {current_theta_gd:>7.4f} | {grad_E_gd:>7.4f} | {delta_theta_k_gd:>7.4f} | {dL_k_val_gd:>7.4f} | {L_total_hist_gd[-1]:>7.4f} | {current_energy_gd:>7.4f}")

#     if not converged_S_total_gd and L_total_hist_gd[-1] >= S_total_geom:
#         N_actual_S_total_gd = k_gd
#         converged_S_total_gd = True
#         print(f"  GD: Converged by S_total at iter {k_gd}")
#     if not converged_energy_gd and np.abs(current_energy_gd - target_energy_gd) < energy_tolerance_gd:
#         if k_gd > 1 and np.abs(energy_hist_gd[-2] - target_energy_gd) >= energy_tolerance_gd :
#             N_actual_energy_gd = k_gd
#             converged_energy_gd = True
#             print(f"  GD: Converged by energy at iter {k_gd}")

#     if np.abs(grad_E_gd) < 1e-5 and k_gd > 5:
#         print(f"  GD: Gradient small ({grad_E_gd:.2e}), stopping early at iter {k_gd}.")
#         if not converged_S_total_gd: N_actual_S_total_gd = k_gd
#         if not converged_energy_gd: N_actual_energy_gd = k_gd
#         break
# # Final check if max_iterations reached
# if not converged_S_total_gd: N_actual_S_total_gd = max_iterations_gd if L_total_hist_gd[-1] < S_total_geom else N_actual_S_total_gd
# if not converged_energy_gd: N_actual_energy_gd = max_iterations_gd if np.abs(energy_hist_gd[-1] - target_energy_gd) >= energy_tolerance_gd else N_actual_energy_gd

# # --- Simulation 2: Gradient Descent with CAPPED step size by delta_theta_max_assumption ---
# print("\n--- Simulation 2: Gradient Descent (parameter step capped by delta_theta_max_assumption) ---")
# eta_capped = 1.0 # Larger learning rate, step will be capped by delta_theta_max
# theta_0_capped = 0.1
# max_iterations_capped = 200

# theta_hist_capped = [theta_0_capped]
# dLk_hist_capped = [0.0]
# L_total_hist_capped = [0.0]
# energy_hist_capped = [energy_function(theta_0_capped)]
# actual_delta_theta_capped_hist = [0.0]


# N_actual_S_total_capped = -1
# N_actual_energy_capped = -1
# converged_S_total_capped = False
# converged_energy_capped = False

# current_theta_capped = theta_0_capped
# print(f"{'Iter':>4} | {'Theta':>7} | {'GradE':>7} | {'dTheta':>7} | {'dL_k':>7} | {'L_total':>7} | {'Energy':>7}")
# print("-" * 60)

# for k_capped in range(1, max_iterations_capped + 1):
#     grad_E_capped = gradient_energy_function(current_theta_capped)
#     delta_theta_k_optimizer_step = -eta_capped * grad_E_capped # Optimizer's proposed step

#     # Cap the step by delta_theta_max_assumption
#     delta_theta_k_actual_capped = np.clip(delta_theta_k_optimizer_step, -delta_theta_max_assumption, delta_theta_max_assumption)
#     actual_delta_theta_capped_hist.append(delta_theta_k_actual_capped)

#     dL_k_val_capped = np.sqrt(g11_fsm(current_theta_capped)) * np.abs(delta_theta_k_actual_capped)

#     current_theta_capped += delta_theta_k_actual_capped

#     theta_hist_capped.append(current_theta_capped)
#     dLk_hist_capped.append(dL_k_val_capped)
#     L_total_hist_capped.append(L_total_hist_capped[-1] + dL_k_val_capped)
#     current_energy_capped = energy_function(current_theta_capped)
#     energy_hist_capped.append(current_energy_capped)

#     print(f"{k_capped:>4} | {current_theta_capped:>7.4f} | {grad_E_capped:>7.4f} | {delta_theta_k_actual_capped:>7.4f} | {dL_k_val_capped:>7.4f} | {L_total_hist_capped[-1]:>7.4f} | {current_energy_capped:>7.4f}")

#     if not converged_S_total_capped and L_total_hist_capped[-1] >= S_total_geom:
#         N_actual_S_total_capped = k_capped
#         converged_S_total_capped = True
#         print(f"  CAPPED: Converged by S_total at iter {k_capped}")
#     if not converged_energy_capped and np.abs(current_energy_capped - target_energy_gd) < energy_tolerance_gd:
#         if k_capped > 1 and np.abs(energy_hist_capped[-2] - target_energy_gd) >= energy_tolerance_gd :
#             N_actual_energy_capped = k_capped
#             converged_energy_capped = True
#             print(f"  CAPPED: Converged by energy at iter {k_capped}")

#     if np.abs(grad_E_capped) < 1e-5 and k_capped > 5 :
#         print(f"  CAPPED: Gradient small ({grad_E_capped:.2e}), stopping early at iter {k_capped}.")
#         if not converged_S_total_capped: N_actual_S_total_capped = k_capped
#         if not converged_energy_capped: N_actual_energy_capped = k_capped
#         break
# # Final check if max_iterations reached
# if not converged_S_total_capped: N_actual_S_total_capped = max_iterations_capped if L_total_hist_capped[-1] < S_total_geom else N_actual_S_total_capped
# if not converged_energy_capped: N_actual_energy_capped = max_iterations_capped if np.abs(energy_hist_capped[-1] - target_energy_gd) >= energy_tolerance_gd else N_actual_energy_capped


# # --- Results Comparison ---
# print("\n--- FINAL RESULTS COMPARISON ---")
# print(f"Theoretical N_min lower bound >= {np.ceil(N_min_theoretical_bound):.0f} (derived assuming optimizer steps are AT MOST delta_theta_max = {delta_theta_max_assumption})")

# print("\nResults for Simulation 1 (Fixed Learning Rate GD):")
# print(f"  Actual N for L_total >= S_total: {N_actual_S_total_gd}")
# print(f"  Actual N for energy convergence: {N_actual_energy_gd}")
# avg_abs_delta_theta_gd = np.mean(np.abs(np.diff(theta_hist_gd[:N_actual_S_total_gd+1]))) if N_actual_S_total_gd > 0 else 0
# print(f"  Average |delta_theta_k| taken by GD optimizer: {avg_abs_delta_theta_gd:.4f}")
# if N_actual_S_total_gd >= np.ceil(N_min_theoretical_bound):
#     print("  Verification: N_actual_S_total (GD) >= N_min_bound (Bound holds)")
# else:
#     print("  Verification: N_actual_S_total (GD) < N_min_bound (Bound might appear violated if avg optimizer step was effectively > assumed delta_theta_max)")

# print("\nResults for Simulation 2 (CAPPED Gradient Descent by delta_theta_max):")
# print(f"  Actual N for L_total >= S_total: {N_actual_S_total_capped}")
# print(f"  Actual N for energy convergence: {N_actual_energy_capped}")
# avg_abs_delta_theta_capped = np.mean(np.abs(actual_delta_theta_capped_hist[1:N_actual_S_total_capped+1])) if N_actual_S_total_capped > 0 else 0 # Use recorded actual steps
# print(f"  Average actual |delta_theta_k| taken by CAPPED optimizer: {avg_abs_delta_theta_capped:.4f} (should be <= {delta_theta_max_assumption})")
# if N_actual_S_total_capped >= np.ceil(N_min_theoretical_bound):
#     print("  Verification: N_actual_S_total (CAPPED) >= N_min_bound (Bound HOLDS as expected)")
# else:
#     print("  Verification: N_actual_S_total (CAPPED) < N_min_bound (Bound appears violated - UNEXPECTED if capping worked)")


# # --- Plotting ---
# # Plot for Simulation 2 (Capped Step Size) as it's more relevant to the bound's assumption
# fig_capped, axs_capped = plt.subplots(3, 1, figsize=(10, 15), sharex=True)

# # Iteration numbers for x-axis
# iters_capped = np.arange(len(theta_hist_capped))

# axs_capped[0].plot(iters_capped, theta_hist_capped, marker='.', linestyle='-')
# axs_capped[0].axhline(np.pi, color='r', linestyle='--', label=f'Target Theta ({np.pi:.2f})')
# axs_capped[0].set_ylabel('Parameter Theta')
# axs_capped[0].set_title(f'VQE (Capped Step, $\Delta\Theta_{{max}}$={delta_theta_max_assumption}) Parameter Evolution')
# axs_capped[0].legend()
# axs_capped[0].grid(True)

# axs_capped[1].plot(iters_capped, L_total_hist_capped, marker='.', linestyle='-')
# axs_capped[1].axhline(S_total_geom, color='r', linestyle='--', label=f'S_total ({S_total_geom:.2f})')
# if N_actual_S_total_capped != -1 and N_actual_S_total_capped <= max_iterations_capped : # Only draw if converged within sim
#     axs_capped[1].axvline(N_actual_S_total_capped, color='g', linestyle=':', label=f'N_actual_S_total ({N_actual_S_total_capped})')
# axs_capped[1].set_ylabel('Cumulative Path Length L_total')
# axs_capped[1].set_title('Cumulative State Manifold Path Length (Capped Step)')
# axs_capped[1].legend()
# axs_capped[1].grid(True)

# axs_capped[2].plot(iters_capped, energy_hist_capped, marker='.', linestyle='-')
# axs_capped[2].axhline(-1.0, color='r', linestyle='--', label='Target Energy (-1.0)')
# axs_capped[2].set_xlabel('Iteration')
# axs_capped[2].set_ylabel('Energy E(theta)')
# axs_capped[2].set_title('VQE Energy Evolution (Capped Step)')
# axs_capped[2].legend()
# axs_capped[2].grid(True)

# plt.tight_layout()
# plt.savefig("vqe_1qubit_qsl_verification_capped_colab.png")
# plt.show()